# Goalkeeper Positioning — 02: Model and Training

`DangerCNN` architecture, classification metrics, the training loop with checkpointing and early stopping, and held-out evaluation against the StatsBomb xG baseline.

**Sections:**
1. Setup
2. `DangerCNN` architecture
3. Metrics
4. Training loop
5. Run training
6. Evaluation

This notebook replaces `src/models/danger_cnn.py`, `src/training/metrics.py`, `src/training/train.py`, and `src/training/evaluate.py`.

**Prerequisites:** Notebook 01 has been run (splits exist under `data/processed/splits/`).

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import log_loss, roc_auc_score
from torch.utils.data import DataLoader

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Import the data primitives defined in notebook 01 (mirrored as src/ modules)
from src.data.dataset import GoalkeeperShotsDataset
from src.data.rasterize import filter_rasterizable_shots

SHOTS_PATH = REPO_ROOT / "data/raw/shots_master_df.csv"
FREEZE_PATH = REPO_ROOT / "data/raw/freeze_master_df.csv"
SPLITS_DIR = REPO_ROOT / "data/processed/splits"
ALL_SPLIT_NAMES = ("train", "val", "test", "transfer_women", "transfer_men_other")

print(f"REPO_ROOT  = {REPO_ROOT}")
print(f"PyTorch    = {torch.__version__}")
print(f"CUDA       = {torch.cuda.is_available()}")
print(f"MPS        = {torch.backends.mps.is_available()}")

## 2. `DangerCNN` architecture

A small (~102k params, 5-channel input) CNN: 3 conv blocks → adaptive average pool → FC head. Designed to land in the 100k–200k parameter range — large enough to capture spatial patterns, small enough to not overfit ~30k shots.

| layer | output |
|---|---|
| input | (B, 5, 80, 60) |
| conv1+bn1+relu+pool | (B, 32, 40, 30) |
| conv2+bn2+relu+pool | (B, 64, 20, 15) |
| conv3+bn3+relu | (B, 128, 20, 15) |
| adaptive_avg_pool2d(1) | (B, 128, 1, 1) → (B, 128) |
| fc1+relu+dropout(0.3) | (B, 64) |
| fc2 | (B, 1) → (B,) |

Use `forward_logits(x)` for training (paired with `BCEWithLogitsLoss` for numerical stability) and `forward(x)` for inference (returns sigmoid probabilities).

### 2.1 Model class

In [ ]:
class DangerCNN(nn.Module):
    def __init__(self, in_channels: int = 5, dropout_p: float = 0.3) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.fc1 = nn.Linear(128, 64)
        self.dropout = nn.Dropout(dropout_p)
        self.fc2 = nn.Linear(64, 1)
        self._init_weights()

    def _init_weights(self) -> None:
        # He (Kaiming) for Conv2d -> ReLU; Xavier for Linear.
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward_logits(self, x: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x.squeeze(-1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(self.forward_logits(x))

### 2.2 Param breakdown

In [ ]:
torch.manual_seed(0)
model = DangerCNN()

print(f"{'Parameter':<28}{'Shape':<22}{'Count':>12}")
print("-" * 62)
total = 0
for name, p in model.named_parameters():
    n = p.numel()
    total += n
    print(f"{name:<28}{str(tuple(p.shape)):<22}{n:>12,}")
print("-" * 62)
print(f"{'TOTAL':<28}{'':<22}{total:>12,}")
in_range = 100_000 <= total <= 200_000
print(f"\n{total:,} params  ({'within' if in_range else 'OUTSIDE'} target 100k–200k range)")

### 2.3 Forward-pass smoke test

In [ ]:
x = torch.randn(4, 5, 80, 60)
model.eval()
with torch.no_grad():
    probs = model(x)
    logits = model.forward_logits(x)

assert probs.shape == (4,)
assert logits.shape == (4,)
assert (probs >= 0).all() and (probs <= 1).all()

print(f"forward(x):        shape={tuple(probs.shape)}  range=[{probs.min():.3f}, {probs.max():.3f}]")
print(f"forward_logits(x): shape={tuple(logits.shape)}  range=[{logits.min():.3f}, {logits.max():.3f}]")

### 2.4 Gradient check

Forward + backward with `BCEWithLogitsLoss` to verify every parameter receives a finite, non-zero gradient (catches dead branches like an init that zeros things out).

In [ ]:
model.train()
logits = model.forward_logits(x)
target = torch.randint(0, 2, (4,), dtype=torch.float32)
loss = nn.BCEWithLogitsLoss()(logits, target)
loss.backward()

problems = []
for name, p in model.named_parameters():
    if p.grad is None:
        problems.append(f"{name} (no grad)")
    elif not torch.isfinite(p.grad).all():
        problems.append(f"{name} (non-finite grad)")
    elif p.grad.abs().sum().item() == 0:
        problems.append(f"{name} (zero grad)")

n_param_tensors = sum(1 for _ in model.parameters())
if problems:
    print(f"FAILED: {problems}")
else:
    print(f"OK: all {n_param_tensors} parameter tensors have finite, non-zero grads (loss={loss.item():.4f})")

### 2.5 Approximate FLOPs per shot

In [ ]:
def conv_flops(c_in, c_out, h_out, w_out, k):
    return 2 * c_in * c_out * h_out * w_out * k * k

flops = (
    conv_flops(5, 32, 80, 60, 3)
    + conv_flops(32, 64, 40, 30, 3)
    + conv_flops(64, 128, 20, 15, 3)
    + 2 * 128 * 64
    + 2 * 64 * 1
)
print(f"Approx FLOPs per shot (forward, batch=1): {flops:,} (~{flops / 1e6:.1f} MFLOPs)")

## 3. Metrics

We report AUC, Brier, log-loss, accuracy at 0.5, and 10-bin equal-width ECE. Brier and ECE are the two we care about most for goalkeeping use cases — calibrated probabilities matter more than ranking.

### 3.1 Expected calibration error

In [ ]:
def expected_calibration_error(y_true, y_pred_probs, n_bins=10):
    """Equal-width-binning ECE: weighted average gap between bin accuracy and bin confidence."""
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_idx = np.clip(np.digitize(y_pred_probs, edges[1:-1]), 0, n_bins - 1)
    n = len(y_true)
    ece = 0.0
    for b in range(n_bins):
        mask = bin_idx == b
        if not mask.any():
            continue
        ece += (mask.sum() / n) * abs(y_true[mask].mean() - y_pred_probs[mask].mean())
    return float(ece)

### 3.2 `compute_metrics`

In [ ]:
def compute_metrics(y_true, y_pred_probs):
    y_true = np.asarray(y_true).astype(float)
    p = np.asarray(y_pred_probs).astype(float)
    p_clip = np.clip(p, 1e-7, 1.0 - 1e-7)
    return {
        "auc": float(roc_auc_score(y_true, p)),
        "brier": float(np.mean((p - y_true) ** 2)),
        "log_loss": float(log_loss(y_true, p_clip)),
        "accuracy_at_0.5": float(np.mean(y_true == (p >= 0.5).astype(float))),
        "expected_calibration_error": expected_calibration_error(y_true, p),
    }

### 3.3 Synthetic-data sanity check

In [ ]:
rng = np.random.default_rng(0)
y_true_syn = (rng.random(1000) < 0.1).astype(float)
y_pred_syn = np.clip(0.1 + 0.4 * y_true_syn + 0.1 * rng.standard_normal(1000), 0.01, 0.99)
syn = compute_metrics(y_true_syn, y_pred_syn)
print("Synthetic-data metrics (n=1000, ~10% positives):")
for k, v in syn.items():
    print(f"  {k:>30s}: {v:.4f}")

## 4. Training loop

`BCEWithLogitsLoss` with **no `pos_weight`** so predictions stay calibrated to the ~10% goal base rate (Brier and ECE remain meaningful for comparison with StatsBomb xG and Anzer & Bauer 2021). Adam + `ReduceLROnPlateau` on val loss + early stopping after `early_stopping_patience` epochs without improvement.

Per run, artifacts go to `models/checkpoints/<run_name>/`:
- `config.json` — exact config used
- `history.json` — per-epoch metrics (updated every epoch)
- `best.pt` — lowest-val-loss checkpoint
- `last.pt` — most recent checkpoint
- `training_curves.png` — train/val loss + val AUC + val Brier

### 4.1 Helpers: seeds, device, manifest cleaning

In [ ]:
def _set_seeds(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _select_device(spec: str) -> torch.device:
    if spec != "auto":
        return torch.device(spec)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def _ensure_clean_manifests(shots_df, freeze_df, split_paths):
    """Drop unrasterizable shots from each split CSV in place. Idempotent."""
    for path in split_paths:
        if not path.exists():
            print(f"  {path.name}: not found, skipping")
            continue
        manifest = pd.read_csv(path)
        kept, dropped = filter_rasterizable_shots(
            manifest["id"].tolist(), shots_df, freeze_df
        )
        if not dropped:
            print(f"  {path.name}: already clean ({len(kept):,} shots)")
            continue
        cleaned = manifest[manifest["id"].isin(set(kept))].copy()
        cleaned.to_csv(path, index=False)
        print(f"  {path.name}: removed {len(dropped):,} unrasterizable, "
              f"kept {len(kept):,}/{len(manifest):,}")

### 4.2 Build train/val DataLoaders

In [ ]:
def _build_dataloaders(config, shots_df, freeze_df):
    cache_train = config["cache_train_in_memory"]
    train_ds = GoalkeeperShotsDataset(
        SPLITS_DIR / "train_shot_ids.csv",
        shots_df, freeze_df, cache_in_memory=cache_train,
    )
    # If we're caching train, val (~615 MB) is a small extra cost and keeps
    # validation from dominating per-epoch wall time.
    val_ds = GoalkeeperShotsDataset(
        SPLITS_DIR / "val_shot_ids.csv",
        shots_df, freeze_df, cache_in_memory=cache_train,
    )
    common = {
        "batch_size": config["batch_size"],
        "num_workers": config["num_workers"],
        "pin_memory": torch.cuda.is_available(),
        "persistent_workers": config["num_workers"] > 0,
    }
    train_loader = DataLoader(train_ds, shuffle=True, **common)
    val_loader = DataLoader(val_ds, shuffle=False, **common)
    return train_loader, val_loader

### 4.3 Validation loop

**Note on the `.cpu()` ordering.** We capture labels on CPU *before* the device transfer. On MPS with `non_blocking=True`, reading `.cpu()` on a tensor that was just moved to-device while other work is queued has been observed to return garbage (verified failure mode on torch 2.11). Capturing `y` on CPU first sidesteps it.

In [ ]:
def _run_validation(model, loader, device, criterion):
    model.eval()
    losses_weighted = 0.0
    all_y, all_p = [], []
    with torch.no_grad():
        for x, y in loader:
            all_y.append(y.numpy())  # <- capture on CPU before .to(device)
            x = x.to(device, non_blocking=True)
            y = y.to(device)
            logits = model.forward_logits(x)
            losses_weighted += criterion(logits, y).item() * x.size(0)
            all_p.append(torch.sigmoid(logits).cpu().numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_p)
    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = losses_weighted / y_true.size
    return metrics

### 4.4 Checkpoint save and curves plot

In [ ]:
def _save_checkpoint(path, model, epoch, val_loss, config):
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "val_loss": val_loss,
            "config": config,
        },
        path,
    )


def _plot_curves(history, path):
    epochs = [h["epoch"] for h in history]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(epochs, [h["train_loss"] for h in history], label="train")
    axes[0].plot(epochs, [h["val_loss"] for h in history], label="val")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE loss")
    axes[0].set_title("Loss"); axes[0].legend()

    axes[1].plot(epochs, [h["val_auc"] for h in history], color="C2")
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("val AUC")
    axes[1].set_title("Validation AUC"); axes[1].set_ylim(0.5, 1.0)

    axes[2].plot(epochs, [h["val_brier"] for h in history], color="C3")
    axes[2].set_xlabel("epoch"); axes[2].set_ylabel("val Brier")
    axes[2].set_title("Validation Brier")

    fig.tight_layout()
    fig.savefig(path, dpi=100)
    plt.close(fig)

### 4.5 `train_model`

In [ ]:
def train_model(config: dict) -> dict:
    """Train DangerCNN per `config`. Returns history + best epoch + checkpoint dir."""
    _set_seeds(config["seed"])
    device = _select_device(config["device"])
    print(f"Device: {device}")

    print("Loading shots and freeze master CSVs...")
    shots_df = pd.read_csv(SHOTS_PATH)
    freeze_df = pd.read_csv(FREEZE_PATH)

    print("Verifying manifests are rasterization-clean:")
    _ensure_clean_manifests(
        shots_df, freeze_df,
        [SPLITS_DIR / f"{n}_shot_ids.csv" for n in ALL_SPLIT_NAMES],
    )

    print("Building DataLoaders...")
    train_loader, val_loader = _build_dataloaders(config, shots_df, freeze_df)

    model = DangerCNN().to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: DangerCNN, {n_params:,} params")

    criterion = nn.BCEWithLogitsLoss()
    if config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=config["learning_rate"],
            weight_decay=config["weight_decay"],
        )
    else:
        raise ValueError(f"Unsupported optimizer: {config['optimizer']!r}")

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5
    )

    ckpt_dir = Path(config["checkpoint_dir"]) / config["run_name"]
    if not ckpt_dir.is_absolute():
        ckpt_dir = REPO_ROOT / ckpt_dir
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    with open(ckpt_dir / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    history: list[dict] = []
    best_val_loss = float("inf")
    best_epoch = -1
    epochs_without_improvement = 0
    log_every = config["log_every_n_batches"]

    for epoch in range(1, config["epochs"] + 1):
        epoch_start = time.perf_counter()
        model.train()
        running_loss = 0.0
        running_n = 0
        for batch_idx, (x, y) in enumerate(train_loader):
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model.forward_logits(x)
            loss = criterion(logits, y)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
            running_n += x.size(0)
            if log_every and (batch_idx + 1) % log_every == 0:
                print(f"  epoch {epoch} batch {batch_idx + 1}/{len(train_loader)} "
                      f"running_loss={running_loss / running_n:.4f}")

        train_loss = running_loss / running_n
        val_metrics = _run_validation(model, val_loader, device, criterion)
        scheduler.step(val_metrics["loss"])
        elapsed = time.perf_counter() - epoch_start
        lr = optimizer.param_groups[0]["lr"]

        record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_auc": val_metrics["auc"],
            "val_brier": val_metrics["brier"],
            "val_log_loss": val_metrics["log_loss"],
            "val_accuracy_at_0.5": val_metrics["accuracy_at_0.5"],
            "val_ece": val_metrics["expected_calibration_error"],
            "lr": lr,
            "epoch_time_s": elapsed,
        }
        history.append(record)
        with open(ckpt_dir / "history.json", "w") as f:
            json.dump(history, f, indent=2)

        print(
            f"epoch {epoch:>3}/{config['epochs']} "
            f"train_loss={train_loss:.4f} "
            f"val_loss={val_metrics['loss']:.4f} "
            f"val_auc={val_metrics['auc']:.4f} "
            f"val_brier={val_metrics['brier']:.4f} "
            f"val_ece={val_metrics['expected_calibration_error']:.4f} "
            f"lr={lr:.2e} ({elapsed:.1f}s)"
        )

        _save_checkpoint(ckpt_dir / "last.pt", model, epoch, val_metrics["loss"], config)
        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_epoch = epoch
            epochs_without_improvement = 0
            _save_checkpoint(ckpt_dir / "best.pt", model, epoch, val_metrics["loss"], config)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= config["early_stopping_patience"]:
                print(f"Early stopping at epoch {epoch}: no val_loss improvement for "
                      f"{config['early_stopping_patience']} epochs "
                      f"(best was epoch {best_epoch}, val_loss={best_val_loss:.4f}).")
                break

    _plot_curves(history, ckpt_dir / "training_curves.png")

    return {
        "history": history,
        "best_epoch": best_epoch,
        "best_val_loss": best_val_loss,
        "checkpoint_dir": str(ckpt_dir),
    }

## 5. Run training

Default config matches `baseline_v1` from the README. Tweak `run_name` to keep new runs separate.

### 5.1 Configuration

In [ ]:
config = {
    "epochs": 50,
    "batch_size": 64,
    "learning_rate": 1e-3,
    "weight_decay": 1e-5,
    "optimizer": "adam",
    "early_stopping_patience": 10,
    "device": "auto",
    "checkpoint_dir": "models/checkpoints",
    "run_name": "baseline_v1",
    "log_every_n_batches": 0,
    "cache_train_in_memory": True,
    "num_workers": 0,
    "seed": 42,
}
config

### 5.2 Train

In [ ]:
result = train_model(config)
print(f"\nTraining complete. Best epoch: {result['best_epoch']} "
      f"(val_loss {result['best_val_loss']:.4f}). "
      f"Artifacts in {result['checkpoint_dir']}.")

## 6. Evaluation

Run a saved checkpoint on a held-out split, compute metrics for both the model and StatsBomb's `shot_statsbomb_xg` on the same shots, and write per-shot predictions to `results/eval/<run_name>/<split>_predictions.csv`.

Valid splits: `test`, `transfer_women`, `transfer_men_other`. The two `transfer_*` splits should only be touched once a final model is chosen.

### 6.1 `evaluate_model`

In [ ]:
VALID_SPLITS = ("test", "transfer_women", "transfer_men_other")


def _select_device_eval():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def evaluate_model(checkpoint_path, split_name):
    if split_name not in VALID_SPLITS:
        raise ValueError(f"split_name must be one of {VALID_SPLITS}, got {split_name!r}")
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.is_absolute():
        checkpoint_path = REPO_ROOT / checkpoint_path

    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    cfg = ckpt.get("config", {})
    run_name = cfg.get("run_name", checkpoint_path.parent.name)

    device = _select_device_eval()
    model = DangerCNN().to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Loaded checkpoint {checkpoint_path.name} from epoch {ckpt['epoch']} "
          f"(val_loss {ckpt['val_loss']:.4f}); device={device}")

    print(f"Loading masters and building loader for split={split_name}...")
    shots_df = pd.read_csv(SHOTS_PATH)
    freeze_df = pd.read_csv(FREEZE_PATH)
    manifest_path = SPLITS_DIR / f"{split_name}_shot_ids.csv"
    ds = GoalkeeperShotsDataset(manifest_path, shots_df, freeze_df, cache_in_memory=False)
    loader = DataLoader(ds, batch_size=cfg.get("batch_size", 64), shuffle=False, num_workers=0)

    all_y, all_p = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device, non_blocking=True)
            logits = model.forward_logits(x)
            all_y.append(y.numpy())
            all_p.append(torch.sigmoid(logits).cpu().numpy())
    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_p)

    xg_lookup = shots_df.set_index("id")["shot_statsbomb_xg"]
    y_xg = xg_lookup.loc[ds.shot_ids].to_numpy()
    valid_xg = ~np.isnan(y_xg)

    model_metrics = compute_metrics(y_true, y_pred)
    xg_metrics = compute_metrics(y_true[valid_xg], y_xg[valid_xg]) if valid_xg.any() else None

    out_dir = REPO_ROOT / "results/eval" / run_name
    out_dir.mkdir(parents=True, exist_ok=True)
    pred_df = pd.DataFrame({
        "id": ds.shot_ids,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_statsbomb_xg": y_xg,
    })
    predictions_path = out_dir / f"{split_name}_predictions.csv"
    pred_df.to_csv(predictions_path, index=False)
    print(f"Saved {len(pred_df):,} predictions to {predictions_path}")

    return {
        "split": split_name,
        "n_shots": len(y_true),
        "n_with_xg": int(valid_xg.sum()),
        "model_metrics": model_metrics,
        "statsbomb_xg_metrics": xg_metrics,
        "predictions_path": str(predictions_path),
    }

### 6.2 Run on `test` split

In [ ]:
test_result = evaluate_model(Path(result["checkpoint_dir"]) / "best.pt", split_name="test")

print(f"\nTest set ({test_result['n_shots']:,} shots, {test_result['n_with_xg']:,} with xG):")
print(f"  {'metric':>30s}  {'DangerCNN':>10s}  {'StatsBomb xG':>14s}")
if test_result["statsbomb_xg_metrics"] is not None:
    for k in ("auc", "brier", "log_loss", "accuracy_at_0.5", "expected_calibration_error"):
        print(f"  {k:>30s}  {test_result['model_metrics'][k]:>10.4f}  "
              f"{test_result['statsbomb_xg_metrics'][k]:>14.4f}")
else:
    for k, v in test_result["model_metrics"].items():
        print(f"  {k:>30s}  {v:>10.4f}  {'(xG missing)':>14s}")

### 6.3 Reference: `baseline_v1` numbers from the README

| metric | DangerCNN | StatsBomb xG |
|---|---|---|
| AUC | 0.8032 | 0.8209 |
| Brier | 0.0750 | 0.0698 |
| ECE | 0.0097 | 0.0079 |

23 epochs, early-stopped, best at epoch 13. Mean prediction (0.098) tracks the test base rate (0.096); decile reliability matches predicted probabilities to within ~1.5 pp across all bins. We're slightly behind StatsBomb xG on every metric — expected, since they have access to many more features (assist type, body part, technique, time pressure). The point of `DangerCNN` isn't to beat xG on aggregate; it's to expose `V(x, g)` as a function of the goalkeeper's position so we can sweep `g` (next notebook).

---

**Done.** A trained checkpoint sits at `models/checkpoints/<run_name>/best.pt`. Notebook 03 uses it to compute the counterfactual `V(x, g)` heatmap.